In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os

# ============= НАСТРОЙКИ =============
statistics_dir = 'C:/Users/Maks/Desktop/Jupyter/Statistics'
output_plots_dir = 'C:/Users/Maks/Desktop/Jupyter/Statistics/plots/heatmaps_v2'
os.makedirs(output_plots_dir, exist_ok=True)
os.makedirs(os.path.join(output_plots_dir, 'U'), exist_ok=True)
os.makedirs(os.path.join(output_plots_dir, 'V'), exist_ok=True)
os.makedirs(os.path.join(output_plots_dir, 'Total_speed'), exist_ok=True)

# Загрузка данных
df = pd.read_csv(os.path.join(statistics_dir, 'wind_statistics_all.csv'))

# Преобразование JSON-строк обратно в списки
for col in ['altitudes', 'u', 'v', 'wind_speed']:
    df[col] = df[col].apply(json.loads)

print(f"Загружено данных: {len(df)} кластеров")

# ============= ПОДГОТОВКА ДАННЫХ ДЛЯ ТЕПЛОВОЙ КАРТЫ =============
print("\nПодготовка данных для тепловых карт...")

# Параметры для анализа
params = [
    'last_flash_amp', 
    'mean_power_until_match', 
    'mean_power_15min',
    'total_power_until_match',    
    'total_power_15min',           
    'total_count_until_match', 
    'count_15min', 
    'cluster_lifetime_minutes'
]

# Названия параметров для подписей
param_names = {
    'last_flash_amp': 'Мощность последнего разряда (A)',
    'mean_power_until_match': 'Средняя мощность (все разряды, A)',
    'mean_power_15min': 'Средняя мощность (15 мин, A)',
    'total_power_until_match': 'Суммарная мощность (все разряды, A)',
    'total_power_15min': 'Суммарная мощность (15 мин, A)',
    'total_count_until_match': 'Количество разрядов (всего)',
    'count_15min': 'Количество разрядов (15 мин)',
    'cluster_lifetime_minutes': 'Время жизни кластера (мин)'
}

# Названия ветровых компонент
wind_components = {
    'wind_speed': 'Total_speed',
    'u': 'U',
    'v': 'V'
}

# Создаем общую сетку высот
all_altitudes = sorted(set([alt for sublist in df['altitudes'] for alt in sublist]))
print(f"Диапазон высот: {min(all_altitudes)} - {max(all_altitudes)} км")
print(f"Количество уникальных высот: {len(all_altitudes)}")

# Для каждого параметра и ветровой компоненты создаем матрицу данных
heatmap_data = {}

for param in params:
    heatmap_data[param] = {}
    for wind_comp in ['wind_speed', 'u', 'v']:
        print(f"\nОбработка {param} - {wind_comp}")
        
        # Собираем данные
        param_values = []
        altitude_matrix = []
        wind_values = []
        
        for idx, row in df.iterrows():
            altitudes = row['altitudes']
            wind_vals = row[wind_comp]
            if param == 'last_flash_amp':
                param_value = abs(row[param])
            else:
                param_value = row[param]
            
            if pd.notna(param_value):
                for alt, wind in zip(altitudes, wind_vals):
                    param_values.append(param_value)
                    altitude_matrix.append(alt)
                    wind_values.append(wind)
        
        temp_df = pd.DataFrame({
            'param': param_values,
            'altitude': altitude_matrix,
            'wind': wind_values
        })
        
        if len(temp_df) > 0:
            unique_params = temp_df['param'].unique()
            n_unique = len(unique_params)
            
            if n_unique <= 15 and all(isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer()) for x in unique_params if pd.notna(x)):
                param_bins = sorted(unique_params)
                param_bins = [param_bins[0] - 0.5] + param_bins + [param_bins[-1] + 0.5]
                param_labels = [f'{param_bins[i]:.0f}' for i in range(1, len(param_bins)-1)]
                temp_df['param_bin'] = pd.cut(temp_df['param'], bins=param_bins, 
                                              labels=param_labels, include_lowest=True)
            else:
                n_bins = min(8, n_unique)
                param_bins = np.percentile(temp_df['param'], np.linspace(0, 100, n_bins + 1))
                param_bins = np.unique(param_bins)
                
                if len(param_bins) < 2:
                    param_bins = [temp_df['param'].min() - 0.001, temp_df['param'].max() + 0.001]
                    param_labels = [f'{param_bins[0]:.1f}']
                else:
                    param_bins[0] = param_bins[0] - 0.001
                    param_bins[-1] = param_bins[-1] + 0.001
                    mid_values = [(param_bins[i] + param_bins[i+1]) / 2 for i in range(len(param_bins)-1)]
                    param_labels = [f'{mid:.1f}' for mid in mid_values]
                
                temp_df['param_bin'] = pd.cut(temp_df['param'], bins=param_bins, 
                                              labels=param_labels, include_lowest=True)
            
            grouped = temp_df.groupby(['param_bin', 'altitude'], observed=False)['wind'].median().reset_index()
            
            param_labels = grouped['param_bin'].unique()
            altitude_labels = sorted(grouped['altitude'].unique())
            
            matrix = np.full((len(param_labels), len(altitude_labels)), np.nan)
            
            for i, p_label in enumerate(param_labels):
                for j, a_label in enumerate(altitude_labels):
                    value = grouped[(grouped['param_bin'] == p_label) & 
                                   (grouped['altitude'] == a_label)]['wind'].values
                    if len(value) > 0:
                        matrix[i, j] = value[0]
            
            heatmap_data[param][wind_comp] = {
                'matrix': matrix,
                'param_labels': param_labels,
                'altitude_labels': altitude_labels
            }

# ============= ПОСТРОЕНИЕ ТЕПЛОВЫХ КАРТ =============
print("\nПостроение тепловых карт...")

colors = [
    (0.5, 0.0, 0.5),  # фиолетовый
    (0, 0, 1),        # синий
    (0, 0.5, 1),      # голубой
    (0.5, 1, 0.5),    # салатовый
    (1, 1, 0),        # желтый
    (1, 0.5, 0),      # оранжевый
    (1, 0, 0),        # красный
    (0.5, 0, 0)       # темно-красный
]
cmap = plt.cm.colors.LinearSegmentedColormap.from_list('custom', colors, N=256)

for param in params:
    for wind_comp, comp_name in wind_components.items():
        if wind_comp in heatmap_data[param]:
            data = heatmap_data[param][wind_comp]
            
            if len(data['matrix']) > 0 and data['matrix'].shape[0] > 1 and data['matrix'].shape[1] > 1:
                fig, ax = plt.subplots(figsize=(10, 10))
                
                matrix_transposed = data['matrix'].T
                
                if comp_name == 'Total_speed':
                    vmin, vmax = 0, 200
                elif comp_name == 'U':
                    vmin, vmax = -150, 150
                else:  # V
                    vmin, vmax = -150, 150
                
                im = ax.imshow(matrix_transposed, aspect='auto', cmap=cmap, 
                              interpolation='nearest', origin='lower',
                              vmin=vmin, vmax=vmax)
                
                ax.set_yticks(np.arange(len(data['altitude_labels'])))
                ax.set_xticks(np.arange(len(data['param_labels'])))
                ax.set_yticklabels([f'{int(alt)}' for alt in data['altitude_labels']], fontsize=26)
                ax.set_xticklabels(data['param_labels'], fontsize=26, rotation=45, ha='right')
                
                ax.set_ylabel('Высота (км)', fontsize=34)
                
                if comp_name == 'Total_speed':
                    ylabel = 'Скорость ветра (м/с)'
                elif comp_name == 'U':
                    ylabel = 'U (м/с)'
                else:
                    ylabel = 'V (м/с)'
                
                ax.set_xlabel(param_names.get(param, param), fontsize=34)
                
                cbar = plt.colorbar(im, ax=ax)
                cbar.set_label(ylabel, fontsize=30)
                cbar.ax.tick_params(labelsize=26)
                
                ax.set_yticks(np.arange(-0.5, len(data['altitude_labels']), 1), minor=True)
                ax.set_xticks(np.arange(-0.5, len(data['param_labels']), 1), minor=True)
                ax.grid(which='minor', color='white', linestyle='-', linewidth=0.5)
                
                plt.tight_layout()
                output_path = os.path.join(output_plots_dir, comp_name, f'heatmap_{param}_{comp_name}.png')
                plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.close()
                
                print(f"  Сохранена: {output_path}")

# ============= КОМБИНИРОВАННАЯ ТЕПЛОВАЯ КАРТА КОРРЕЛЯЦИЙ =============
print("\nСоздание комбинированной тепловой карты корреляций...")

for wind_comp, comp_name in wind_components.items():
    correlation_by_altitude = pd.DataFrame(index=all_altitudes, columns=params)
    
    for alt in all_altitudes:
        for param in params:
            wind_at_alt = []
            param_values = []
            
            for _, row in df.iterrows():
                altitudes = row['altitudes']
                if alt in altitudes:
                    idx = altitudes.index(alt)
                    wind_at_alt.append(row[wind_comp][idx])
                    param_values.append(row[param])
            
            if len(wind_at_alt) > 3:
                corr = np.corrcoef(param_values, wind_at_alt)[0, 1]
                correlation_by_altitude.loc[alt, param] = corr if not np.isnan(corr) else 0
            else:
                correlation_by_altitude.loc[alt, param] = 0
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    corr_matrix = correlation_by_altitude.values.astype(float)
    
    im = ax.imshow(corr_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest')
    
    ax.set_xticks(np.arange(len(params)))
    ax.set_yticks(np.arange(len(all_altitudes)))
    ax.set_xticklabels([param_names.get(p, p).replace(' ', '\n') for p in params], rotation=45, ha='right', fontsize=30)
    ax.set_yticklabels([f'{int(alt)}' for alt in all_altitudes], fontsize=30)
    
    if comp_name == 'Total_speed':
        title = 'Корреляция параметров гроз со скоростью ветра по высотам'
    elif comp_name == 'U':
        title = 'Корреляция параметров гроз с U компонентой ветра по высотам'
    else:
        title = 'Корреляция параметров гроз с V компонентой ветра по высотам'
    
    ax.set_xlabel('Параметры', fontsize=26)
    ax.set_ylabel('Высота (км)', fontsize=26)
    ax.set_title(title, fontsize=30, fontweight='bold')
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Коэффициент корреляции', fontsize=24)
    
    for i in range(len(params)):
        for j in range(len(all_altitudes)):
            value = corr_matrix[j, i]
            if not np.isnan(value) and abs(value) > 0.1:
                text_color = 'white' if abs(value) > 0.6 else 'black'
                ax.text(i, j, f'{value:.2f}', ha='center', va='center', 
                       color=text_color, fontsize=10)
    
    plt.tight_layout()
    output_path = os.path.join(output_plots_dir, comp_name, f'correlation_heatmap_{comp_name}.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Сохранена корреляционная карта для {comp_name}: {output_path}")

# ============= ВЫВОД СТАТИСТИКИ =============
print("\n" + "="*60)
print("СТАТИСТИКА ТЕПЛОВЫХ КАРТ")
print("="*60)

print(f"\nВсе тепловые карты сохранены в: {output_plots_dir}")
print("  - Папка Total_speed: графики для общей скорости ветра")
print("  - Папка U: графики для U компоненты")
print("  - Папка V: графики для V компоненты")
print("\nОбработка завершена!")

Загружено данных: 54 кластеров

Подготовка данных для тепловых карт...
Диапазон высот: 82.5 - 117.5 км
Количество уникальных высот: 15

Обработка last_flash_amp - wind_speed

Обработка last_flash_amp - u

Обработка last_flash_amp - v

Обработка mean_power_until_match - wind_speed

Обработка mean_power_until_match - u

Обработка mean_power_until_match - v

Обработка mean_power_15min - wind_speed

Обработка mean_power_15min - u

Обработка mean_power_15min - v

Обработка total_power_until_match - wind_speed

Обработка total_power_until_match - u

Обработка total_power_until_match - v

Обработка total_power_15min - wind_speed

Обработка total_power_15min - u

Обработка total_power_15min - v

Обработка total_count_until_match - wind_speed

Обработка total_count_until_match - u

Обработка total_count_until_match - v

Обработка count_15min - wind_speed

Обработка count_15min - u

Обработка count_15min - v

Обработка cluster_lifetime_minutes - wind_speed

Обработка cluster_lifetime_minutes - u